# Самостоятельная работа. Анализ датасета russian_demography.csv

**Практическая работа №1 · Технологии и инструментарий анализа больших данных**

Датасет: демографические показатели регионов России (1990–2017).


## 1. Постановка задачи

**Источник.** Файл `russian_demography.csv` — открытый агрегированный набор  
демографической статистики по субъектам РФ. Аналогичные показатели  
публикует Росстат (естественное движение населения, урбанизация и др.).

**Предметная область.** Естественный прирост, рождаемость, смертность,  
демографическая нагрузка и доля городского населения по регионам и годам.

**Цель анализа.** Выполнить полный базовый цикл работы с табличными данными:  
загрузка и проверка структуры, оценка качества, предобработка,  
описательные статистики и групповые срезы, визуализация (включая временной ряд),  
экспорт подготовленного набора и формулировка выводов.

**Ограничения набора.** Данные агрегированы по году и региону;  
персональных данных нет; у части регионов в отдельные годы имеются пропуски.


## 2. Проверка рабочего окружения

Фиксация версий интерпретатора и библиотек необходима для воспроизводимости  
результатов: другой запуск или другое окружение могут дать иной вывод  
из‑за различий в версиях pandas/numpy.


In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

# Берём только номер версии Python (без строки платформы)
print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Matplotlib:", matplotlib.__version__)


## 3. Загрузка данных

CSV читается со стандартным разделителем «,» и кодировкой UTF-8.  
Типы столбцов pandas определяет автоматически при чтении.


In [ ]:
from pathlib import Path

# Path — объект пути, независимый от особенностей ОС
DATA_PATH = Path("russian_demography.csv")

# read_csv возвращает DataFrame; первая строка файла — заголовок
df = pd.read_csv(DATA_PATH)

# shape: (число строк, число столбцов)
print(df.shape)

# Список имён столбцов
print(df.columns.tolist())

# Первые строки — быстрая проверка корректности загрузки
df.head()


Таблица содержит 2380 наблюдений и 7 признаков.  
Если бы столбцы «склеились» или число строк отличалось от ожидаемого,  
потребовалось бы явно задать `sep` или `encoding`.


## 4. Паспорт данных

| Поле | Исходный dtype | Смысл |
|------|----------------|-------|
| `year` | int | Год наблюдения (1990–2017). Временной признак. |
| `region` | object | Название субъекта РФ. Категориальный признак. |
| `npg` | float | Natural Population Growth — естественный прирост, ‰. |
| `birth_rate` | float | Коэффициент рождаемости, ‰. |
| `death_rate` | float | Коэффициент смертности, ‰. |
| `gdw` | float | Коэффициент демографической нагрузки  
| | | (число нетрудоспособных на 100 трудоспособных). |
| `urbanization` | float | Доля городского населения, %. |

По смыслу `npg ≈ birth_rate − death_rate`; это соотношение  
можно проверить на данных. Признак `region` категориальный,  
`year` задаёт временную ось для последующего ряда.


## 5. Первичная диагностика данных

Базовый набор проверок:
- `info()` — структура, типы, оценка памяти;
- `isna()`, `duplicated()`, `nunique()` — пропуски, дубликаты, кардинальность;
- `describe()` — описательные статистики.

Эти методы решают разные задачи и не подменяют друг друга.


### 5.1. Структура и память


In [ ]:
import io

# StringIO — текстовый буфер в памяти; info() пишет в него,
# а не сразу в вывод ячейки (удобно для воспроизводимого отчёта)
buf = io.StringIO()
df.info(buf=buf, memory_usage="deep")  # deep учитывает размер object-строк
print(buf.getvalue())


В выводе `info` важно отметить dtype столбцов и наличие `object`  
у `region` — кандидат на преобразование в `category`.


### 5.2. Пропуски, дубликаты, уникальные значения


In [ ]:
# isna() → булева таблица; sum() считает True как 1
print("Пропуски по столбцам:")
print(df.isna().sum())

# Полностью совпадающие строки
print("\nПолных дубликатов:", df.duplicated().sum())

# Число различных значений по каждому столбцу
print("\nЧисло уникальных значений:")
print(df.nunique())


Пропуски присутствуют в `npg`, `birth_rate`, `death_rate`, `gdw`, `urbanization`  
(десятки строк; в основном отдельные регионы в отдельные годы —  
Ингушетия, Крым, Севастополь и др.). Полных дубликатов нет.  

`region` — 85 уникальных значений при 2380 строках, поэтому преобразование  
в `category` даст экономию памяти. `year` принимает 28 значений (1990–2017).


### 5.3. Описательная статистика


In [ ]:
# include='all' — статистики и для числовых, и для object
# .T — транспонирование: признаки в строках, удобнее читать
print(df.describe(include='all').round(2).T)


Средний `npg` отрицательный (примерно −2,3 ‰): в среднем по выборке  
наблюдается естественная убыль. Средняя рождаемость ниже средней смертности.  
`urbanization` меняется от ~24 % до 100 % — сильный региональный разброс.  
Распределение `npg` асимметрично: есть правый хвост регионов с высоким приростом.


## 6. Типы данных и предобработка

Задачи этого раздела:
1. привести типы к смыслу данных;
2. зафиксировать решение по пропускам;
3. создать вычисляемый признак.


### 6.1. Оптимизация типов


In [ ]:
# deep=True — реальный размер object-строк
before = df.memory_usage(deep=True).sum()

# Независимая копия, чтобы не менять исходный df
clean_df = df.copy()

# category: словарь уникальных значений + компактные коды
# вместо 2380 повторяющихся строк
clean_df["region"] = clean_df["region"].astype("category")

# Диапазон 1990–2017 укладывается в int16
clean_df["year"] = clean_df["year"].astype("int16")

after = clean_df.memory_usage(deep=True).sum()
print(f"До: {before / 1024**2:.2f} MiB")
print(f"После: {after / 1024**2:.2f} MiB")
print(f"Снижение: {(1 - after / before) * 100:.1f}%")


Преобразование `region` в `category` эффективно, когда число уникальных  
значений невелико относительно числа строк. Для `year` достаточно `int16`.


### 6.2. Пропуски

Пропуски не заполняются медианой или средним по всей таблице:  
они сосредоточены у конкретных регионов в конкретные годы,  
а подстановка общероссийского среднего исказила бы региональную картину.  

Для большинства расчётов достаточно локального исключения строк с NaN  
(`dropna` в нужном месте).


In [ ]:
# any(axis=1) — True, если в строке есть хотя бы один пропуск
print("Строк с хотя бы одним пропуском:", clean_df.isna().any(axis=1).sum())
print("Строк без пропусков:", clean_df.dropna().shape[0])


### 6.3. Вычисляемый признак `is_decline`

`is_decline` — индикатор естественной убыли населения (`npg < 0`).  
Признак удобен для фильтрации и подсчёта долей.

Дополнительно проверяется согласованность:  
`npg` должен быть близок к `birth_rate − death_rate`.


In [ ]:
# Векторное сравнение по всем строкам
clean_df["is_decline"] = clean_df["npg"] < 0

# Только строки, где заполнены все три поля
mask = clean_df[['npg', 'birth_rate', 'death_rate']].notna().all(axis=1)

# Абсолютное расхождение |birth − death − npg|
diff = (
    clean_df.loc[mask, 'birth_rate']
    - clean_df.loc[mask, 'death_rate']
    - clean_df.loc[mask, 'npg']
).abs()

# max() возвращает скаляр float — используем встроенный round()
print("Макс. |birth − death − npg|:", round(float(diff.max()), 4))
print("Доля наблюдений с |diff| > 0.15:", round(float((diff > 0.15).mean()), 4))

print(clean_df[["year", "region", "npg", "birth_rate", "death_rate", "is_decline"]].head(8))


Если максимальное расхождение мало (порядка 0,1 и меньше),  
`npg` действительно соответствует разности рождаемости и смертности  
с учётом округления. Существенные расхождения были бы сигналом  
о проблеме качества данных.


## 7. Фильтрация

Составное условие: естественная убыль (`npg < 0`) и высокая урбанизация  
(> 80 %). Условие учебное — иллюстрирует работу `loc` с несколькими  
критериями, а не задаёт нормативный порог.


In [ ]:
# loc[маска строк, список столбцов]
# Каждое сравнение — в скобках; между ними & (поэлементное И).
# Оператор and здесь не подходит: он работает с одиночными True/False,
# а не с Series.
subset = clean_df.loc[
    (clean_df['npg'] < 0) & (clean_df['urbanization'] > 80),
    ['year', 'region', 'npg', 'birth_rate', 'death_rate', 'urbanization']
]

print("Строк в срезе:", len(subset))
print(subset.sort_values('npg').head(10))


Срез выделяет регионы и годы с убылью населения при высокой доле  
городского населения. Число строк в результате показывает,  
насколько жёстким оказалось выбранное условие.


## 8. Группировка и агрегирование

Фильтрация отбирает строки по условию.  
Группировка разбивает данные на группы и вычисляет агрегаты внутри каждой.

Три среза:
1. распределение по `is_decline`;
2. средние показатели по годам;
3. рейтинг регионов по среднему `npg`.


### 8.1. Доля наблюдений с убылью и приростом


In [ ]:
# value_counts — частоты; dropna=False сохраняет NaN как отдельную категорию
# to_frame — Series → DataFrame; assign добавляет столбец с долей в %
decline_stats = (
    clean_df['is_decline']
      .value_counts(dropna=False)
      .to_frame('n')
      .assign(share_pct=lambda x: x['n'] / len(clean_df) * 100)
)
print(decline_stats.round(2))


Доля `True` показывает, насколько типична естественная убыль  
в рассматриваемом наборе наблюдений.


### 8.2. Средние показатели по годам


In [ ]:
# groupby + agg с именованными агрегатами:
# новое_имя=(исходный_столбец, функция)
# observed=True — не создавать пустые группы для category
yearly = (
    clean_df.groupby('year', observed=True)
      .agg(
          n=('region', 'size'),
          npg_mean=('npg', 'mean'),
          birth_mean=('birth_rate', 'mean'),
          death_mean=('death_rate', 'mean'),
          urban_mean=('urbanization', 'mean'),
          gdw_mean=('gdw', 'mean')
      )
)
print(yearly.round(2).head(10))
print('...')
print(yearly.round(2).tail(5))


Одним проходом считаются и объём группы (`n`), и средние значения.  
Сравнение средних имеет смысл только с учётом `n`: оценка по десяткам  
регионов надёжнее, чем по нескольким.


### 8.3. Рейтинг регионов по среднему npg


In [ ]:
region_npg = (
    clean_df.groupby('region', observed=True)['npg']
      .agg(['mean', 'median', 'count'])
      .sort_values('mean', ascending=False)
)
print('Топ-10 по среднему npg:')
print(region_npg.head(10).round(2))
print('\nХудшие 10:')
print(region_npg.tail(10).round(2))


В верхней части рейтинга обычно оказываются республики Северного Кавказа,  
Тыва, Алтай; в нижней — области Центральной России и Северо-Запада.  
Столбец `count` показывает, по скольким годам рассчитано среднее.


## 9. Корреляции и потенциальные выбросы

Коэффициент корреляции Пирсона оценивает линейную связь (−1…1).  
Значение около нуля означает отсутствие выраженной линейной зависимости,  
но не отсутствие любой связи и не причинность.

Правило IQR помечает наблюдения за пределами  
[Q1 − 1,5·IQR; Q3 + 1,5·IQR] как кандидатов на проверку,  
а не как автоматические ошибки.


### 9.1. Корреляционная матрица


In [ ]:
num_cols = ['npg', 'birth_rate', 'death_rate', 'gdw', 'urbanization']

# corr() по умолчанию считает попарный коэффициент Пирсона
corr = clean_df[num_cols].corr()
print(corr.round(3))


`npg` сильно положительно связан с `birth_rate` и отрицательно с `death_rate` —  
это ожидаемо. `urbanization` умеренно отрицательно коррелирует с `npg`  
и `birth_rate`: более урбанизированные регионы чаще характеризуются убылью.  
Слабые коэффициенты не интерпретируются как устойчивый эффект.


### 9.2. Правило IQR для death_rate


In [ ]:
q1 = clean_df['death_rate'].quantile(0.25)
q3 = clean_df['death_rate'].quantile(0.75)
iqr = q3 - q1

# Границы правила Тьюки
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

# Выход за нижнюю или верхнюю границу
mask = (clean_df['death_rate'] < lower) | (clean_df['death_rate'] > upper)

print(f"Q1={q1:.2f}; Q3={q3:.2f}; IQR={iqr:.2f}")
print(f"Границы: [{lower:.2f}; {upper:.2f}]")
print("Кандидатов:", int(mask.sum()),
      f"({100 * mask.mean():.2f}% строк)")


Доля помеченных наблюдений умеренная. Высокая смертность в отдельные  
годы и в отдельных регионах — известный демографический феномен  
(особенно 1990-е), поэтому автоматическое удаление таких строк  
не применяется: IQR лишь формирует список для предметной проверки.


## 10. Визуальный разведочный анализ

Выбор типа графика определяется аналитическим вопросом:

| Вопрос | Тип графика |
|--------|-------------|
| Сравнение категорий | столбчатая диаграмма |
| Форма распределения одного числа | гистограмма |
| Связь двух числовых признаков | диаграмма рассеяния |
| Динамика во времени | линейный график (временной ряд) |

Поле `year` присутствует, поэтому четвёртый график — временной ряд.


### 10.1. Средний npg по регионам (топ-15 и худшие-15)


In [ ]:
top = region_npg.head(15)['mean']
bottom = region_npg.tail(15)['mean']
plot_data = pd.concat([top, bottom]).sort_values()

fig, ax = plt.subplots(figsize=(9, 7))

# Цвет зависит от знака значения
colors = ['#d62728' if v < 0 else '#2ca02c' for v in plot_data]
plot_data.plot(kind='barh', ax=ax, color=colors)

ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Средний естественный прирост (‰) по регионам\n(топ-15 и худшие-15 за 1990–2017)')
ax.set_xlabel('Средний npg, ‰')
ax.set_ylabel('Регион')
plt.tight_layout()
plt.show()


График подчёркивает региональный контраст: устойчивый прирост  
на Кавказе и в Тыве против устойчивой убыли в центре и на северо-западе.


### 10.2. Распределение npg


In [ ]:
plt.figure(figsize=(8, 4.5))

# dropna — гистограмма не принимает NaN
plt.hist(clean_df['npg'].dropna(), bins=35, edgecolor='white')

plt.axvline(0, color='red', linestyle='--', label='npg = 0')
plt.title('Распределение естественного прироста населения')
plt.xlabel('npg, ‰')
plt.ylabel('Частота')
plt.legend()
plt.tight_layout()
plt.show()


Основная масса наблюдений левее нуля (убыль), имеется правый хвост  
регионов с высоким приростом. Гистограмма подтверждает асимметрию,  
на которую указывал `describe`.


### 10.3. Рождаемость и смертность


In [ ]:
# Подвыборка, чтобы не перегружать график; random_state для воспроизводимости
ok = clean_df.dropna(subset=['birth_rate', 'death_rate'])
sample = ok.sample(n=min(2000, len(ok)), random_state=42)

plt.figure(figsize=(8, 5))
plt.scatter(sample['birth_rate'], sample['death_rate'], s=12, alpha=0.4)

# Диагональ birth = death
plt.plot([5, 30], [5, 30], 'r--', linewidth=1, label='birth = death')

plt.title('Рождаемость и смертность')
plt.xlabel('birth_rate, ‰')
plt.ylabel('death_rate, ‰')
plt.legend()
plt.tight_layout()
plt.show()


Точки выше диагонали соответствуют случаям, когда смертность  
превышает рождаемость. Облако широкое: жёсткой линейной зависимости нет,  
но общая тенденция прослеживается.


### 10.4. Временной ряд средних показателей


In [ ]:
# Используем таблицу yearly, построенную в разделе 8.2
ts = yearly[['birth_mean', 'death_mean', 'npg_mean']].dropna()

plt.figure(figsize=(10, 5))
plt.plot(ts.index, ts['birth_mean'], marker='o', markersize=3, label='Рождаемость')
plt.plot(ts.index, ts['death_mean'], marker='o', markersize=3, label='Смертность')
plt.plot(ts.index, ts['npg_mean'], marker='o', markersize=3, label='Естественный прирост')

plt.axhline(0, color='gray', linestyle='--', linewidth=0.8)
plt.title('Динамика средних демографических показателей (1990–2017)')
plt.xlabel('Год')
plt.ylabel('‰')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


На ряде видны характерные периоды:
- 1990-е — падение рождаемости и рост смертности, выраженная убыль;
- 2000-е — частичное восстановление рождаемости;
- после середины 2010-х — новое снижение рождаемости.

Именно наличие поля `year` делает построение временного ряда корректным.


## 11. Экспорт подготовленных данных

Один и тот же `clean_df` сохраняется в трёх форматах,  
чтобы сравнить размер файлов и поведение при повторном чтении.

| Формат | Сильная сторона | Ограничение |
|--------|-----------------|-------------|
| CSV | универсальный обмен | типы восстанавливаются заново |
| JSON Lines | построчные объекты | больший размер для таблиц |
| Parquet | сохранение типов, сжатие, колоночное чтение | нужен движок (pyarrow) |


In [ ]:
from pathlib import Path

OUT = Path("output_demography")
OUT.mkdir(exist_ok=True)

# index=False — не записывать технический индекс в файл
clean_df.to_csv(
    OUT / 'russian_demography_clean.csv',
    index=False,
    encoding='utf-8'
)

# JSON Lines: один объект на строку; force_ascii=False сохраняет кириллицу
clean_df.to_json(
    OUT / 'russian_demography_clean.json',
    orient='records',
    lines=True,
    force_ascii=False
)

clean_df.to_parquet(
    OUT / 'russian_demography_clean.parquet',
    index=False,
    engine='pyarrow',
    compression='snappy'
)

files = [
    OUT / 'russian_demography_clean.csv',
    OUT / 'russian_demography_clean.json',
    OUT / 'russian_demography_clean.parquet'
]
for path in files:
    size_mib = path.stat().st_size / 1024**2
    print(f'{path.name:40s} {size_mib:7.2f} MiB')


### Проверка повторного чтения


In [ ]:
csv_back = pd.read_csv(OUT / 'russian_demography_clean.csv')
parquet_back = pd.read_parquet(OUT / 'russian_demography_clean.parquet')

print('CSV shape:', csv_back.shape)
print('Parquet shape:', parquet_back.shape)
print('\nParquet dtypes:')
print(parquet_back.dtypes)


Число строк и столбцов совпадает с сохранённой таблицей.  
Parquet сохраняет `category` и `int16`.  
CSV при повторном чтении заново выводит типы: `region` снова становится  
`object`, `year` — `int64`. В этом одно из практических отличий форматов.


## 12. Итоговые выводы и ограничения

### Выводы
1. Набор содержит 2380 строк и 7 исходных признаков. Полных дубликатов нет;  
   пропуски (порядка 2–3 % строк) сосредоточены у отдельных регионов в отдельные годы.
2. Средний естественный прирост по выборке отрицательный (≈ −2,3 ‰):  
   убыль — преобладающий режим наблюдений.
3. Региональный контраст устойчив: прирост характерен для республик Северного Кавказа,  
   Тывы и Алтая; убыль — для многих областей Центральной России и Северо-Запада.
4. Преобразование `region` в `category` и `year` в `int16` снижает объём таблицы  
   в памяти без потери информации.
5. `npg` согласован с разностью `birth_rate − death_rate`; урбанизация умеренно  
   отрицательно связана с рождаемостью и приростом.
6. Временной ряд отражает демографический спад 1990-х, частичное восстановление  
   в 2000-х и новое снижение рождаемости после середины 2010-х.
7. Правило IQR помечает часть значений `death_rate`, однако эти наблюдения  
   отражают реальную дифференциацию и не удаляются автоматически.

### Ограничения
1. Данные агрегированы по году и субъекту РФ — внутрирегиональная дифференциация  
   и сезонность недоступны.
2. Пропуски не заполнялись: нет надёжного основания для импутации без искажения  
   региональной картины; часть регионов представлена неполно.
3. Корреляции и средние не устанавливают причинно-следственных связей  
   (миграция, возрастная структура, экономические факторы в файле отсутствуют).
4. Сопоставимость показателей во времени может зависеть от изменений  
   методологии учёта.


## 13. Ответы на контрольные вопросы

1. **Числовой dtype не гарантирует, что признак количественный.**  
   Код категории (например, числовой идентификатор региона) хранится как число,  
   но среднее по нему не имеет содержательного смысла.

2. **DataFrame и ndarray.**  
   DataFrame хранит именованные столбцы, допускает смешанные типы и индекс.  
   ndarray — однородный массив без схемы столбцов.

3. **head, info, describe решают разные задачи.**  
   `head` показывает примеры строк, `info` — структуру, типы и память,  
   `describe` — статистики. Одна функция не покрывает все три потребности.

4. **Отсутствие пропусков и дубликатов не означает отсутствие проблем качества.**  
   Возможны ошибочные значения, несогласованность кодов и названий,  
   смена методологии учёта, выбросы.

5. **`category` уменьшает память**, когда уникальных значений мало,  
   а строк много: вместо повторяющихся строк хранятся коды и словарь категорий.

6. **Фильтрация и группировка.**  
   Фильтрация отбирает строки по условию.  
   Группировка разбивает данные на группы и считает агрегаты внутри каждой.

7. **Коэффициент корреляции около нуля** означает отсутствие выраженной  
   *линейной* связи. Из этого нельзя сделать вывод об отсутствии любой зависимости  
   и нельзя утверждать причинность.

8. **Правило IQR** — эвристика для поиска кандидатов на проверку.  
   Решение об удалении принимается только после предметной интерпретации.

9. **Выбор типа графика** определяется вопросом: сравнение категорий —  
   столбчатая диаграмма; форма распределения — гистограмма;  
   связь двух чисел — диаграмма рассеяния; динамика — временной ряд.

10. **Временной ряд нельзя строить по индексу строк**, если индекс не является  
    временем: порядок загрузки не несёт временной информации.

11. **Parquet** сохраняет типы, поддерживает сжатие и колоночное чтение —  
    это удобнее для аналитического pipeline, чем текстовый CSV.

12. **Команда Run All после перезапуска ядра** проверяет, что notebook  
    самодостаточен: все переменные создаются в правильном порядке,  
    нет скрытых зависимостей от ручного запуска отдельных ячеек.
